# 12 — Data Mart Export (Parquet)

**Tujuan notebook ini (enhancement Fase 12):**
Konsolidasi 5 tabel analytics yang sudah dibangun (notebook 04, 06, 10, 11) sebagai **data mart** siap pakai untuk Power BI — dengan tipe data yang dibersihkan dan disimpan sebagai **Parquet** (bukan CSV), supaya:
- Tipe data (tanggal, boolean, numeric) terjaga presisi saat diimport ke Power BI
- Ukuran file lebih kecil, load lebih cepat (columnar & compressed)

> Ini tetap konsisten dengan alur roadmap Fase 12 (`MongoDB → Python → Processed Tables → Power BI`) — cuma format Processed Tables-nya di-upgrade dari CSV ke Parquet.


In [1]:
import sys
try:
    import pyarrow
except ModuleNotFoundError:
    !{sys.executable} -m pip install pyarrow

import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data_processed")

print("pyarrow siap, lanjut export ke Parquet.")


pyarrow siap, lanjut export ke Parquet.


---
## 1. Load 5 Tabel Data Mart (dari CSV yang sudah ada)


In [2]:
customer_analytics = pd.read_csv(
    DATA_DIR / "customer_analytics.csv",
    parse_dates=["first_order_date", "last_order_date"]
)
product_analytics = pd.read_csv(DATA_DIR / "product_analytics.csv")
seller_analytics = pd.read_csv(DATA_DIR / "seller_analytics.csv")
customer_segmentation = pd.read_csv(DATA_DIR / "customer_segmentation_full.csv")
predictive_analytics = pd.read_csv(DATA_DIR / "dashboard5_predictive_analytics.csv")

tables = {
    "customer_analytics": customer_analytics,
    "product_analytics": product_analytics,
    "seller_analytics": seller_analytics,
    "customer_segmentation": customer_segmentation,
    "predictive_analytics": predictive_analytics,
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")


customer_analytics: 96,096 rows, 8 columns
product_analytics: 32,951 rows, 7 columns
seller_analytics: 3,095 rows, 6 columns
customer_segmentation: 96,096 rows, 7 columns
predictive_analytics: 54,738 rows, 9 columns


---
## 2. Bersihkan Tipe Data Sebelum Export

Pastikan kolom boolean, integer, dan kategori bertipe eksplisit (bukan `object`/string generik) — supaya Power BI membaca tipe yang benar tanpa perlu diubah manual di Power Query.


In [3]:
# customer_segmentation: pastikan 'segment' jadi kategori eksplisit
customer_segmentation["segment"] = customer_segmentation["segment"].astype("category")

# predictive_analytics: pastikan 'prediction' jadi boolean-friendly, 'priority_quadrant' jadi kategori
predictive_analytics["prediction"] = predictive_analytics["prediction"].astype(bool)
predictive_analytics["priority_quadrant"] = predictive_analytics["priority_quadrant"].astype("category")
predictive_analytics["value_band"] = predictive_analytics["value_band"].astype("category")
predictive_analytics["threshold_method"] = predictive_analytics["threshold_method"].astype("category")

# product_analytics & seller_analytics: category untuk kolom state/kategori (opsional, hemat memory)
if "category" in product_analytics.columns:
    product_analytics["category"] = product_analytics["category"].astype("category")
if "seller_state" in seller_analytics.columns:
    seller_analytics["seller_state"] = seller_analytics["seller_state"].astype("category")
if "customer_state" in customer_analytics.columns:
    customer_analytics["customer_state"] = customer_analytics["customer_state"].astype("category")

print("Tipe data sudah dibersihkan.")
print(customer_analytics.dtypes)


Tipe data sudah dibersihkan.
customer_unique_id                   str
total_orders                       int64
total_spending                   float64
avg_order_value                  float64
first_order_date          datetime64[us]
last_order_date           datetime64[us]
customer_lifetime_days             int64
customer_state                  category
dtype: object


---
## 3. Export ke Parquet


In [4]:
for name, df in tables.items():
    output_path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(output_path, engine="pyarrow", index=False)
    csv_size = (DATA_DIR / f"{name}.csv").stat().st_size / 1024 if (DATA_DIR / f"{name}.csv").exists() else None
    parquet_size = output_path.stat().st_size / 1024
    if csv_size:
        print(f"{name}: CSV {csv_size:.1f} KB -> Parquet {parquet_size:.1f} KB "
              f"({(1 - parquet_size/csv_size)*100:.0f}% lebih kecil)")
    else:
        print(f"{name}: Parquet {parquet_size:.1f} KB")


customer_analytics: CSV 8835.8 KB -> Parquet 5406.3 KB (39% lebih kecil)
product_analytics: CSV 2287.7 KB -> Parquet 1360.9 KB (41% lebih kecil)
seller_analytics: CSV 208.3 KB -> Parquet 150.3 KB (28% lebih kecil)
customer_segmentation: Parquet 3692.5 KB
predictive_analytics: Parquet 2522.3 KB


---
## Definition of Done

- [ ] 5 tabel data mart di-load dan tipe datanya dibersihkan
- [ ] Semua tabel berhasil di-export sebagai `.parquet` di `data_processed/`
- [ ] Perbandingan ukuran file CSV vs Parquet terdokumentasi

**Lanjut ke:** import file `.parquet` ini ke Power BI (Get Data → Parquet), bukan lagi CSV.
